In [1]:
# Build a consolidated LPG GeoPackage by merging:
# - manually verified points
# - attractiveness-based API points
#
# Final output layers:
# - resell
# - filling
# - resell_and_filling
# - depot
#
# Main rules:
# - manual points get attractiveness = 1
# - API reseller points inside false_points are excluded
# - duplicates are removed (place_id preferred, geometry fallback)
# - mutual exclusion priority: depot > filling > resell
# - coordinate aliases are normalized (lat/lon)
# - points with invalid friction_walk OR friction_moto are removed from every layer
#
# Final schema additions:
# - stable IDs: id_res&fil, id_depot
# - resell: initialize cost_kg_walker=999 and cost_kg_driver=999
# - filling: initialize cost_kg=999, cost_kg_walker=999, cost_kg_driver=999
# - depot: no cost columns initialized
# - legacy fid is removed

In [2]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio

# Input / output files
DATA_DIR = "dataset_big"

manual_file = f"{DATA_DIR}/manual_check_nig_3857.gpkg"
attrac_file = f"{DATA_DIR}/lpg_points_nigeria_attractiveness.gpkg"
output_file = f"{DATA_DIR}/full_lpg_chain_nig_3857.gpkg"


def standardize_coordinate_columns(gdf):
    """Normalize coordinate column names to avoid duplicates when merging.
    Canonical names: lat, lon"""
    if gdf is None or gdf.empty:
        return gdf

    gdf = gdf.copy()

    # First-pass rename when canonical column is missing
    rename_map = {}
    if 'lng' in gdf.columns and 'lon' not in gdf.columns:
        rename_map['lng'] = 'lon'
    if 'longitude' in gdf.columns and 'lon' not in gdf.columns and 'lng' not in gdf.columns:
        rename_map['longitude'] = 'lon'
    if 'latitude' in gdf.columns and 'lat' not in gdf.columns:
        rename_map['latitude'] = 'lat'

    if rename_map:
        gdf = gdf.rename(columns=rename_map)

    # Second-pass consolidation if aliases still coexist
    if 'lon' in gdf.columns and 'lng' in gdf.columns:
        gdf['lon'] = gdf['lon'].combine_first(gdf['lng'])
        gdf = gdf.drop(columns=['lng'])
    if 'lon' in gdf.columns and 'longitude' in gdf.columns:
        gdf['lon'] = gdf['lon'].combine_first(gdf['longitude'])
        gdf = gdf.drop(columns=['longitude'])
    if 'lat' in gdf.columns and 'latitude' in gdf.columns:
        gdf['lat'] = gdf['lat'].combine_first(gdf['latitude'])
        gdf = gdf.drop(columns=['latitude'])

    return gdf


def resolve_existing_path(candidates, label):
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Could not find {label}. Checked: {candidates}"
    )


def sample_valid_friction_mask(gdf, raster_path):
    if gdf is None or gdf.empty:
        return pd.Series(dtype=bool)

    with rasterio.open(raster_path) as src:
        if gdf.crs != src.crs:
            gdf_local = gdf.to_crs(src.crs)
        else:
            gdf_local = gdf

        coords = []
        for geom in gdf_local.geometry:
            if geom is None or geom.is_empty:
                coords.append((np.nan, np.nan))
            elif geom.geom_type == 'Point':
                coords.append((geom.x, geom.y))
            else:
                c = geom.centroid
                coords.append((c.x, c.y))

        sampled = []
        for arr in src.sample(coords):
            if arr is None or len(arr) == 0:
                sampled.append(np.nan)
            else:
                sampled.append(float(arr[0]))

        values = np.array(sampled, dtype=float)
        valid = np.isfinite(values)
        if src.nodata is not None:
            valid &= values != float(src.nodata)
        valid &= values > 0

    return pd.Series(valid, index=gdf.index)


def filter_layer_by_friction(gdf, layer_name, friction_walk_path, friction_moto_path):
    if gdf is None or gdf.empty:
        return gdf

    valid_walk = sample_valid_friction_mask(gdf, friction_walk_path)
    valid_moto = sample_valid_friction_mask(gdf, friction_moto_path)
    valid_mask = valid_walk & valid_moto

    before_n = len(gdf)
    filtered = gdf.loc[valid_mask].copy()
    removed_n = before_n - len(filtered)
    print(
        f"    {layer_name}: removed {removed_n} outside valid friction "
        f"(kept {len(filtered)}/{before_n})"
    )
    return filtered


# 1. Load manual layers
print("\n[1] Loading manual layers...")
manual_layers = {
    "reseller": "reseller",
    "filling": "filling",
    "primary_depot": "primary_depot",
    "false_points": "false_points"
}
manual_data = {}
for layer_name, layer_key in manual_layers.items():
    try:
        gdf = gpd.read_file(manual_file, layer=layer_key)
        # Remove any existing index_right column to avoid conflicts
        if 'index_right' in gdf.columns:
            gdf = gdf.drop(columns='index_right')
        gdf = standardize_coordinate_columns(gdf)
        print(f"    {layer_key}: {len(gdf)} points")
        manual_data[layer_key] = gdf
    except Exception as e:
        print(f"    Warning: Could not read layer '{layer_key}' from {manual_file}: {e}")
        manual_data[layer_key] = None

# 2. Load attractiveness points
print("\n[2] Loading attractiveness points...")
try:
    attrac_gdf = gpd.read_file(attrac_file, layer='lpg_points')
    if 'index_right' in attrac_gdf.columns:
        attrac_gdf = attrac_gdf.drop(columns='index_right')
    attrac_gdf = standardize_coordinate_columns(attrac_gdf)
    print(f"    lpg_points: {len(attrac_gdf)} points")
    # Ensure CRS is EPSG:3857 for consistency with manual file
    if attrac_gdf.crs != "EPSG:3857":
        attrac_gdf = attrac_gdf.to_crs("EPSG:3857")
        print("    Reprojected to EPSG:3857")
except Exception as e:
    print(f"    Error: Could not load {attrac_file}: {e}")
    raise

# 3. Filter out false points from attractiveness
print("\n[3] Removing false points from attractiveness data...")
false_gdf = manual_data.get("false_points")
if false_gdf is not None and not false_gdf.empty:
    # Ensure same CRS
    if false_gdf.crs != attrac_gdf.crs:
        false_gdf = false_gdf.to_crs(attrac_gdf.crs)
    # Use only geometry to avoid column conflicts
    false_geom = false_gdf[['geometry']].copy()
    # Spatial join: keep only attractiveness points not within any false point
    sjoin = gpd.sjoin(attrac_gdf, false_geom, how='left', predicate='within')
    attrac_valid = sjoin[sjoin['index_right'].isna()].copy()
    attrac_valid = attrac_valid.drop(columns='index_right')
    print(f"    Removed {len(attrac_gdf) - len(attrac_valid)} points that lie inside false points.")
    print(f"    Remaining attractiveness points: {len(attrac_valid)}")
else:
    attrac_valid = attrac_gdf.copy()
    print("    No false_points layer found, keeping all attractiveness points.")

# 4. Prepare manual points: add attractiveness = 1, keep all columns
print("\n[4] Adding attractiveness=1 to manual points...")
def prepare_manual(gdf, name):
    if gdf is None or gdf.empty:
        return None
    gdf = gdf.copy()
    gdf['attractiveness'] = 1.0
    print(f"    {name}: {len(gdf)} points, attractiveness set to 1")
    return gdf

manual_reseller = prepare_manual(manual_data.get("reseller"), "reseller")
manual_filling = prepare_manual(manual_data.get("filling"), "filling")
manual_primary = prepare_manual(manual_data.get("primary_depot"), "primary_depot")

# 5. Build initial layers with duplicate removal using place_id (preferred) or geometry
print("\n[5] Building initial layers (before mutual exclusion)...")
def combine_and_deduplicate(dfs, sources_description):
    """Combine multiple GeoDataFrames and drop duplicates.
       Prefer place_id if available in all sources; fallback to geometry."""
    valid_dfs = [df for df in dfs if df is not None and not df.empty]
    if not valid_dfs:
        return None

    before_count = sum(len(df) for df in valid_dfs)
    combined = pd.concat(valid_dfs, ignore_index=True)

    # Check if all source DataFrames have 'place_id'
    all_have_place_id = all('place_id' in df.columns for df in valid_dfs)

    if all_have_place_id:
        # Deduplicate using place_id (fast and reliable)
        combined = combined.drop_duplicates(subset='place_id', keep='first')
        method = "place_id"
    else:
        # Fallback to geometry (WKT)
        combined['__wkt'] = combined.geometry.apply(lambda g: g.wkt)
        combined = combined.drop_duplicates(subset='__wkt', keep='first')
        combined = combined.drop(columns='__wkt')
        method = "geometry"

    after_count = len(combined)
    combined = gpd.GeoDataFrame(combined, geometry='geometry', crs=valid_dfs[0].crs)
    print(f"    {sources_description}: {before_count} points before dedup -> {after_count} points after dedup (using {method})")
    return combined

# Initial layers
depot_gdf = manual_primary  # depot only from manual
resell_gdf = combine_and_deduplicate([manual_reseller, attrac_valid], "resell (manual reseller + attractiveness)")
filling_gdf = manual_filling  # filling only from manual

# 6. Apply mutual exclusion based on place_id (priority: depot > filling > resell)
print("\n[6] Applying mutual exclusion (depot > filling > resell)...")
if depot_gdf is not None and not depot_gdf.empty:
    depot_ids = set(depot_gdf['place_id'])
    if filling_gdf is not None and not filling_gdf.empty:
        before_fill = len(filling_gdf)
        filling_gdf = filling_gdf[~filling_gdf['place_id'].isin(depot_ids)]
        print(f"    Removed {before_fill - len(filling_gdf)} filling points that are also in depot.")
    if resell_gdf is not None and not resell_gdf.empty:
        before_resell = len(resell_gdf)
        resell_gdf = resell_gdf[~resell_gdf['place_id'].isin(depot_ids)]
        print(f"    Removed {before_resell - len(resell_gdf)} resell points that are also in depot.")

if filling_gdf is not None and not filling_gdf.empty:
    filling_ids = set(filling_gdf['place_id'])
    if resell_gdf is not None and not resell_gdf.empty:
        before_resell = len(resell_gdf)
        resell_gdf = resell_gdf[~resell_gdf['place_id'].isin(filling_ids)]
        print(f"    Removed {before_resell - len(resell_gdf)} resell points that are also in filling.")

# 7. Remove points outside valid friction for every base layer
print("\n[7] Removing points outside valid friction (walk + moto)...")
friction_walk_file = resolve_existing_path(
    [
        f"{DATA_DIR}/friction_walk.tif",
        f"{DATA_DIR}/walk_friction.tif",
        f"{DATA_DIR}/Rasters/friction_walk.tif",
        f"{DATA_DIR}/Rasters/walk_friction.tif",
        f"{DATA_DIR}/rasters/friction_walk.tif",
        f"{DATA_DIR}/rasters/walk_friction.tif"
    ],
    label="walk friction raster"
 )
friction_moto_file = resolve_existing_path(
    [
        f"{DATA_DIR}/friction_moto.tif",
        f"{DATA_DIR}/moto_friction.tif",
        f"{DATA_DIR}/Rasters/friction_moto.tif",
        f"{DATA_DIR}/Rasters/moto_friction.tif",
        f"{DATA_DIR}/rasters/friction_moto.tif",
        f"{DATA_DIR}/rasters/moto_friction.tif"
    ],
    label="moto friction raster"
 )
print(f"    walk friction: {friction_walk_file}")
print(f"    moto friction: {friction_moto_file}")

depot_gdf = filter_layer_by_friction(depot_gdf, "depot", friction_walk_file, friction_moto_file)
filling_gdf = filter_layer_by_friction(filling_gdf, "filling", friction_walk_file, friction_moto_file)
resell_gdf = filter_layer_by_friction(resell_gdf, "resell", friction_walk_file, friction_moto_file)

# 8. Build resell_and_filling from the final resell and filling layers
print("\n[8] Building resell_and_filling layer...")
resell_fill_gdf = combine_and_deduplicate([resell_gdf, filling_gdf], "resell_and_filling (final resell + final filling)")

# 9. Add stable ID columns and remove legacy fid attribute
print("\n[9] Creating stable ID columns...")

def _point_key_series(gdf):
    if gdf is None or gdf.empty:
        return pd.Series(dtype='object')
    if 'place_id' in gdf.columns:
        pid = gdf['place_id'].astype(str).fillna('')
        has_pid = pid.str.len() > 0
        wkt = gdf.geometry.apply(lambda geom: geom.wkt)
        return pd.Series(np.where(has_pid, 'pid:' + pid, 'wkt:' + wkt), index=gdf.index)
    wkt = gdf.geometry.apply(lambda geom: geom.wkt)
    return pd.Series('wkt:' + wkt, index=gdf.index)

def _drop_legacy_fid(gdf):
    if gdf is None:
        return None
    gdf = gdf.copy()
    if 'fid' in gdf.columns:
        gdf = gdf.drop(columns=['fid'])
    return gdf

def _set_cost_kg(gdf, default_value=999):
    if gdf is None or gdf.empty:
        return gdf
    gdf = gdf.copy()
    gdf['cost_kg'] = float(default_value)
    return gdf

def _set_cost_walker_driver(gdf, default_value=999):
    if gdf is None or gdf.empty:
        return gdf
    gdf = gdf.copy()
    gdf['cost_kg_walker'] = float(default_value)
    gdf['cost_kg_driver'] = float(default_value)
    return gdf

def _move_id_cols_first(gdf):
    if gdf is None or gdf.empty:
        return gdf
    id_cols = [
        c for c in ['id_res&fil', 'id_depot', 'cost_kg', 'cost_kg_walker', 'cost_kg_driver']
        if c in gdf.columns
    ]
    other_cols = [c for c in gdf.columns if c not in id_cols + ['geometry']]
    return gdf[id_cols + other_cols + ['geometry']]

# Drop legacy fid attribute before generating new IDs
resell_gdf = _drop_legacy_fid(resell_gdf)
filling_gdf = _drop_legacy_fid(filling_gdf)
resell_fill_gdf = _drop_legacy_fid(resell_fill_gdf)
depot_gdf = _drop_legacy_fid(depot_gdf)

# id_res&fil: unique progressive ID across resell + filling domain
if resell_fill_gdf is not None and not resell_fill_gdf.empty:
    key_rf = _point_key_series(resell_fill_gdf)
    unique_keys = pd.unique(key_rf)
    id_map = {k: i + 1 for i, k in enumerate(unique_keys)}
    resell_fill_gdf['id_res&fil'] = key_rf.map(id_map).astype('Int64')

    if resell_gdf is not None and not resell_gdf.empty:
        resell_gdf['id_res&fil'] = _point_key_series(resell_gdf).map(id_map).astype('Int64')
    if filling_gdf is not None and not filling_gdf.empty:
        filling_gdf['id_res&fil'] = _point_key_series(filling_gdf).map(id_map).astype('Int64')

# id_depot: progressive ID only for depot
if depot_gdf is not None and not depot_gdf.empty:
    depot_gdf = depot_gdf.reset_index(drop=True)
    depot_gdf['id_depot'] = pd.Series(np.arange(1, len(depot_gdf) + 1), dtype='Int64')

# Ensure all ID columns exist in each layer
for g in [resell_gdf, filling_gdf, resell_fill_gdf, depot_gdf]:
    if g is not None and not g.empty:
        if 'id_res&fil' not in g.columns:
            g['id_res&fil'] = pd.Series([pd.NA] * len(g), dtype='Int64')
        if 'id_depot' not in g.columns:
            g['id_depot'] = pd.Series([pd.NA] * len(g), dtype='Int64')

# Add default cost columns by layer schema
resell_gdf = _set_cost_walker_driver(resell_gdf, default_value=999)
filling_gdf = _set_cost_kg(filling_gdf, default_value=999)
filling_gdf = _set_cost_walker_driver(filling_gdf, default_value=999)
resell_fill_gdf = _set_cost_kg(resell_fill_gdf, default_value=999)
resell_fill_gdf = _set_cost_walker_driver(resell_fill_gdf, default_value=999)

# Move ID columns to the beginning
resell_gdf = _move_id_cols_first(resell_gdf)
filling_gdf = _move_id_cols_first(filling_gdf)
resell_fill_gdf = _move_id_cols_first(resell_fill_gdf)
depot_gdf = _move_id_cols_first(depot_gdf)

print("    IDs created: id_res&fil, id_depot")
print("    Added defaults: resell(cost_kg_walker, cost_kg_driver), filling(cost_kg, cost_kg_walker, cost_kg_driver)")

# 10. Write output GeoPackage
print(f"\n[10] Saving to {output_file}...")
if resell_gdf is not None and not resell_gdf.empty:
    resell_gdf.to_file(output_file, layer="resell", driver="GPKG")
if filling_gdf is not None and not filling_gdf.empty:
    filling_gdf.to_file(output_file, layer="filling", driver="GPKG")
if resell_fill_gdf is not None and not resell_fill_gdf.empty:
    resell_fill_gdf.to_file(output_file, layer="resell_and_filling", driver="GPKG")
if depot_gdf is not None and not depot_gdf.empty:
    depot_gdf.to_file(output_file, layer="depot", driver="GPKG")

# 11. Summary of final counts
print("\n" + "=" * 60)
print("Summary of final layers (after mutual exclusion + friction validity filter):")
print("=" * 60)
print(f"  - depot             : {len(depot_gdf) if depot_gdf is not None else 0} points")
print(f"  - filling           : {len(filling_gdf) if filling_gdf is not None else 0} points")
print(f"  - resell            : {len(resell_gdf) if resell_gdf is not None else 0} points")
print(f"  - resell_and_filling: {len(resell_fill_gdf) if resell_fill_gdf is not None else 0} points")

# 12. Total unique points across all layers (using place_id if possible)
all_points = []
for gdf in [depot_gdf, filling_gdf, resell_gdf, resell_fill_gdf]:
    if gdf is not None and not gdf.empty:
        all_points.append(gdf)

if all_points:
    combined_all = pd.concat(all_points, ignore_index=True)
    if 'place_id' in combined_all.columns:
        total_unique = combined_all['place_id'].nunique()
        print(f"\nTotal unique points across all layers (by place_id): {total_unique}")
    else:
        combined_all['__wkt'] = combined_all.geometry.apply(lambda g: g.wkt)
        total_unique = combined_all['__wkt'].nunique()
        print(f"\nTotal unique points across all layers (by geometry): {total_unique}")
else:
    print("\nNo layers created.")

print("\nDone.")


[1] Loading manual layers...
    reseller: 89 points
    filling: 376 points
    primary_depot: 14 points
    false_points: 187 points

[2] Loading attractiveness points...
    lpg_points: 2833 points

[3] Removing false points from attractiveness data...
    Removed 108 points that lie inside false points.
    Remaining attractiveness points: 2725

[4] Adding attractiveness=1 to manual points...
    reseller: 89 points, attractiveness set to 1
    filling: 376 points, attractiveness set to 1
    primary_depot: 14 points, attractiveness set to 1

[5] Building initial layers (before mutual exclusion)...
    resell (manual reseller + attractiveness): 2814 points before dedup -> 2749 points after dedup (using place_id)

[6] Applying mutual exclusion (depot > filling > resell)...
    Removed 0 filling points that are also in depot.
    Removed 10 resell points that are also in depot.
    Removed 323 resell points that are also in filling.

[7] Removing points outside valid friction (walk 

In [3]:
# 12. Post-process cost schema by layer
#    - resell: cost_kg_walker, cost_kg_driver = 999
#    - filling: cost_kg, cost_kg_walker, cost_kg_driver = 999
#    - depot: remove cost columns

import geopandas as gpd
import pandas as pd

print("\n[12] Harmonizing cost schema per layer...")

RES_COST_WALK = "cost_kg_walker"
RES_COST_CAR = "cost_kg_driver"
FILL_COST = "cost_kg"
DEFAULT_COST = 999.0


def _read_layer_safe(path, layer_name):
    try:
        gdf = gpd.read_file(path, layer=layer_name)
        if gdf is None or gdf.empty:
            return None
        return gdf
    except Exception:
        return None


def _write_layer(path, layer_name, gdf):
    if gdf is not None and (not gdf.empty):
        gdf.to_file(path, layer=layer_name, driver="GPKG")


# Resell layer
resell_out = _read_layer_safe(output_file, "resell")
if resell_out is not None:
    resell_out[RES_COST_WALK] = DEFAULT_COST
    resell_out[RES_COST_CAR] = DEFAULT_COST
    resell_out[RES_COST_WALK] = pd.to_numeric(resell_out[RES_COST_WALK], errors="coerce").astype(float)
    resell_out[RES_COST_CAR] = pd.to_numeric(resell_out[RES_COST_CAR], errors="coerce").astype(float)
    if FILL_COST in resell_out.columns:
        resell_out = resell_out.drop(columns=[FILL_COST])
    _write_layer(output_file, "resell", resell_out)
    print(f"    resell: set {RES_COST_WALK} and {RES_COST_CAR} to {DEFAULT_COST:g}")

# Filling layer
filling_out = _read_layer_safe(output_file, "filling")
if filling_out is not None:
    filling_out[FILL_COST] = DEFAULT_COST
    filling_out[RES_COST_WALK] = DEFAULT_COST
    filling_out[RES_COST_CAR] = DEFAULT_COST
    filling_out[FILL_COST] = pd.to_numeric(filling_out[FILL_COST], errors="coerce").astype(float)
    filling_out[RES_COST_WALK] = pd.to_numeric(filling_out[RES_COST_WALK], errors="coerce").astype(float)
    filling_out[RES_COST_CAR] = pd.to_numeric(filling_out[RES_COST_CAR], errors="coerce").astype(float)
    _write_layer(output_file, "filling", filling_out)
    print(f"    filling: set {FILL_COST}, {RES_COST_WALK}, {RES_COST_CAR} to {DEFAULT_COST:g}")

# Depot layer
depot_out = _read_layer_safe(output_file, "depot")
if depot_out is not None:
    drop_cols = [c for c in [FILL_COST, RES_COST_WALK, RES_COST_CAR] if c in depot_out.columns]
    if drop_cols:
        depot_out = depot_out.drop(columns=drop_cols)
    _write_layer(output_file, "depot", depot_out)
    print("    depot: removed cost columns")

print("[12] Cost schema harmonization done.")


[12] Harmonizing cost schema per layer...
    resell: set cost_kg_walker and cost_kg_driver to 999
    filling: set cost_kg, cost_kg_walker, cost_kg_driver to 999
    depot: removed cost columns
[12] Cost schema harmonization done.
